# Notebook 1 — Trích đặc trưng và khám phá dữ liệu

Quy trình (xem `docs/huong-dan-thu-du-lieu-bao-cao.md`, mục 0.1):

```
Video → 2 khung/giây → YuNet (tìm mặt) → căn chỉnh 112×112 → ArcFace → vector 512 chiều   [có sẵn, không train]
      → SVM / cosine / k-NN / MLP → P1…P5 hoặc NGUOI_LA                                [train]
```

Engine và ngưỡng dùng lại **đúng của app**: `backend/apps/attendance/face/onnx_engine.py` và `FACE` trong `backend/config/settings/base.py`.

Lần đầu chạy mất 5–15 phút. Các lần sau dùng lại `ket_qua/dataset_mat.csv`; muốn trích lại thì xoá file đó. Chạy xong thì mở **`02_train_danh_gia.ipynb`**.

## 1. Cấu hình

In [ ]:
import random
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

GOC = Path.cwd() if (Path.cwd() / "du_lieu").exists() else Path.cwd() / "ml_report"
VIDEO = GOC / "du_lieu" / "video"
LFW = GOC / "du_lieu" / "lfw"
ANH = GOC / "du_lieu" / "anh_mat"  # ảnh mặt đã căn chỉnh (nằm trong du_lieu/ nên không lên git)
KQ = GOC / "ket_qua"
MODELS = GOC.parent / "backend" / "face_models"
KQ.mkdir(exist_ok=True)
ANH.mkdir(parents=True, exist_ok=True)

SEED = 42
KHUNG_MOI_GIAY = 2
CANH_TIM_MAT = 640  # cạnh dài của ảnh đưa vào YuNet
LFW_SO_NGUOI, LFW_ANH_MOI_NGUOI = 150, 3
# Ngưỡng giống app (settings.FACE)
MIN_SANG, MIN_NET, MIN_RONG = 60, 50, 100
NGUONG_COSINE = 0.5  # accept_threshold
LA = "NGUOI_LA"

sys.path.insert(0, str(GOC.parent / "backend"))
from apps.attendance.face.onnx_engine import MAX_SIDE, OnnxFaceEngine, align, yaw_proxy  # noqa: E402

plt.rcParams["figure.dpi"] = 110
print("Thư mục:", GOC)

## 2. Cắt khung hình, tìm mặt, trích đặc trưng

Mỗi khung hình: giữ mặt lớn nhất. Khung bị **loại** nếu không thấy mặt, mặt rộng < 100 px, quá tối (< 60) hoặc nhoè (phương sai Laplacian < 50) — cùng tiêu chí app dùng khi điểm danh. Ảnh LFW và ảnh giả mạo chỉ bị loại khi không thấy mặt.

In [ ]:
E = [f"e{i}" for i in range(512)]
engine = None


def thu_nho(img, canh=MAX_SIDE):
    s = canh / max(img.shape[:2])
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img


def phan_tich(img, loc_chat_luong):
    """Trả (thông tin, ảnh căn chỉnh, khung mặt) hoặc (lý do loại, None, None)."""
    img = thu_nho(img)
    # YuNet mất tự tin với mặt quá to (mặt 545 px chỉ được 0,77 < 0,8): tìm mặt trên ảnh cạnh 640, quy toạ độ về ảnh gốc
    nho = thu_nho(img, CANH_TIM_MAT)
    faces = engine.detect(nho).copy()
    faces[:, :14] *= img.shape[1] / nho.shape[1]
    if len(faces) == 0:
        return "khong_thay_mat", None, None
    f = max(faces, key=lambda f: f[2] * f[3])
    kps = f[4:14].reshape(5, 2)
    al = align(img, kps)
    gray = cv2.cvtColor(al, cv2.COLOR_BGR2GRAY)
    info = dict(
        do_sang=float(gray.mean()),
        do_net=float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        goc_mat=yaw_proxy(kps),
        rong_mat=float(f[2]),
    )
    if loc_chat_luong:
        if info["rong_mat"] < MIN_RONG:
            return "mat_nho", None, None
        if info["do_sang"] < MIN_SANG:
            return "toi", None, None
        if info["do_net"] < MIN_NET:
            return "mo", None, None
    info["diem_that"] = engine._spoof(img, tuple(f[:4]))
    return info, al, (img, f)


def doc_khung(path):
    cap = cv2.VideoCapture(str(path))
    buoc = max(1, round((cap.get(cv2.CAP_PROP_FPS) or 30) / KHUNG_MOI_GIAY))
    i = 0
    while cap.grab():
        if i % buoc == 0:
            ok, img = cap.retrieve()
            if ok:
                yield i, img
        i += 1
    cap.release()


def nguon_anh():
    """Sinh (nguoi, buoi, clip, chỉ số, ảnh)."""
    for v in sorted(VIDEO.glob("P*/*/*")):
        if v.suffix.lower() in (".mp4", ".mov"):
            for i, img in doc_khung(v):
                yield v.parts[-3], v.parts[-2], v.stem, i, img
    if LFW.exists():
        ten = sorted(p.name for p in LFW.iterdir() if p.is_dir())
        for t in random.Random(SEED).sample(ten, min(LFW_SO_NGUOI, len(ten))):
            for j, p in enumerate(sorted((LFW / t).glob("*.jpg"))[:LFW_ANH_MOI_NGUOI]):
                yield LA, "lfw", t, j, cv2.imread(str(p))


def trich_dac_trung():
    global engine
    thieu = [f for f in ("face_detection_yunet_2023mar.onnx", "w600k_r50.onnx", "MiniFASNetV2.onnx", "MiniFASNetV1SE.onnx")
             if not (MODELS / f).exists()]
    assert not thieu, f"Thiếu mô hình ONNX trong {MODELS}: {thieu} (xem mục 6.2 tài liệu)"
    assert any(VIDEO.glob("P*/buoi*/*.*")), f"Không tìm thấy video trong {VIDEO} (kiểm tra tên thư mục, mục 4.1 tài liệu)"
    engine = OnnxFaceEngine(MODELS)
    dong, loai, mau = [], [], {}
    so_khung = sum(
        int(cv2.VideoCapture(str(v)).get(cv2.CAP_PROP_FRAME_COUNT)) // 15 + 1
        for v in VIDEO.glob("P*/*/*.*")
    ) + (LFW_SO_NGUOI * LFW_ANH_MOI_NGUOI if LFW.exists() else 0)  # ước lượng cho thanh tiến độ
    for nguoi, buoi, clip, i, img in tqdm(nguon_anh(), total=so_khung, desc="Trích đặc trưng"):
        that = buoi.startswith("buoi")
        info, al, goc = phan_tich(img, loc_chat_luong=that)
        if al is None:
            loai.append(dict(nguoi=nguoi, buoi=buoi, clip=clip, khung=i, ly_do=info))
            continue
        file = ANH / nguoi / buoi / f"{clip}_{i:04d}.jpg"
        file.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(file), al)
        emb = engine._embedding(al) if buoi != "gia_mao" else np.zeros(512, np.float32)
        dong.append(
            dict(nguoi=nguoi, buoi=buoi, clip=clip, file=file.relative_to(GOC).as_posix(), **info)
            | dict(zip(E, emb))
        )
        if that and nguoi not in mau:
            mau[nguoi] = (goc, al)
    df = pd.DataFrame(dong)
    df[df.buoi == "gia_mao"].drop(columns=E).to_csv(KQ / "gia_mao.csv", index=False)
    df = df[df.buoi != "gia_mao"]
    df.to_csv(KQ / "dataset_mat.csv", index=False, float_format="%.6g")
    pd.DataFrame(loai).to_csv(KQ / "anh_bi_loai.csv", index=False)
    ve_can_chinh(mau)


def ve_can_chinh(mau):
    fig, ax = plt.subplots(2, len(mau), figsize=(2.6 * len(mau), 5.4), squeeze=False)
    for k, (nguoi, ((img, f), al)) in enumerate(sorted(mau.items())):
        x, y, w, h = (int(v) for v in f[:4])
        m = int(0.4 * w)
        crop = img[max(0, y - m) : y + h + m, max(0, x - m) : x + w + m].copy()
        for px, py in f[4:14].reshape(5, 2):
            cv2.circle(crop, (int(px) - max(0, x - m), int(py) - max(0, y - m)), 3, (0, 255, 0), -1)
        ax[0, k].imshow(crop[..., ::-1])
        ax[0, k].set_title(f"{nguoi}: khung gốc + 5 điểm mốc", fontsize=8)
        ax[1, k].imshow(al[..., ::-1])
        ax[1, k].set_title("Sau căn chỉnh 112×112", fontsize=8)
    for a in ax.ravel():
        a.axis("off")
    fig.tight_layout()
    fig.savefig(KQ / "anh_can_chinh.png")
    plt.show()


if not (KQ / "dataset_mat.csv").exists():
    trich_dac_trung()
else:
    print("Dùng lại dataset_mat.csv đã có (xoá file để trích lại).")
    if (KQ / "anh_can_chinh.png").exists():
        plt.figure(figsize=(13, 5.4))
        plt.imshow(plt.imread(KQ / "anh_can_chinh.png"))
        plt.axis("off")
        plt.show()

df = pd.read_csv(KQ / "dataset_mat.csv")
loai = pd.read_csv(KQ / "anh_bi_loai.csv")
gm_path = KQ / "gia_mao.csv"
gia_mao = pd.read_csv(gm_path) if gm_path.exists() and gm_path.stat().st_size > 1 else pd.DataFrame()
print(f"{len(df)} ảnh mặt, {len(loai)} khung bị loại, {len(gia_mao)} ảnh giả mạo")
df.head()

## 3. Thống kê ảnh

Mỗi người, mỗi buổi nên có **từ 40 ảnh** trở lên.

In [ ]:
so_anh = df.pivot_table(index="nguoi", columns="buoi", values="file", aggfunc="count", fill_value=0)
display(so_anh)
thieu = [(n, b) for n in so_anh.index if n != LA for b in ("buoi1", "buoi2", "buoi3") if so_anh.get(b, {}).get(n, 0) < 40]
if thieu:
    print("⚠ Dưới 40 ảnh:", ", ".join(f"{n}/{b}" for n, b in thieu))

ax = so_anh.drop(index=LA, errors="ignore").drop(columns="lfw", errors="ignore").plot.bar(figsize=(7, 3.5), rot=0)
ax.set_ylabel("Số ảnh")
ax.set_title("Số ảnh khuôn mặt của từng người theo buổi")
plt.tight_layout()
plt.savefig(KQ / "so_anh_moi_nguoi.png")
plt.show()

### Ảnh bị loại
Tỉ lệ loại nên dưới khoảng 30%. Cao hơn nghĩa là video mờ, tối hoặc mặt quá nhỏ.

In [ ]:
if len(loai):
    bang_loai = loai.pivot_table(index=["nguoi", "buoi"], columns="ly_do", values="khung", aggfunc="count", fill_value=0)
    giu = df.groupby(["nguoi", "buoi"]).size().rename("giu_lai")
    bang_loai = bang_loai.join(giu, how="outer").fillna(0).astype(int)
    bang_loai["ti_le_loai_%"] = (100 * bang_loai.drop(columns="giu_lai").sum(axis=1) / bang_loai.sum(axis=1)).round(1)
    display(bang_loai)
else:
    print("Không có khung nào bị loại.")

### Thống kê các trường số (MIN, MAX, AVERAGE, COUNT, độ lệch chuẩn)

In [ ]:
COT_SO = ["do_sang", "do_net", "goc_mat", "rong_mat", "diem_that"]
thong_ke = df[COT_SO].agg(["min", "max", "mean", "count", "std"]).T
thong_ke.columns = ["MIN", "MAX", "AVERAGE", "COUNT", "STD"]
thong_ke.round(3).to_csv(KQ / "thong_ke.csv")
display(thong_ke.round(3))

for cot, ten, file in [
    ("do_sang", "Độ sáng trung bình vùng mặt (0–255)", "phan_bo_do_sang.png"),
    ("do_net", "Độ nét (phương sai Laplacian, thang log)", "phan_bo_do_net.png"),
    ("goc_mat", "Góc quay ngang ước lượng (yaw proxy)", "phan_bo_goc_mat.png"),
]:
    fig, ax = plt.subplots(figsize=(6, 3))
    v = df.loc[df.buoi != "lfw", cot].dropna()
    if cot == "do_net":
        v, nhan = np.log10(v.clip(lower=1)), "log10(do_net)"
    elif cot == "goc_mat":  # mặt quay gần nghiêng hẳn làm giá trị vọt tới ±15; dồn về ±2 để thấy phần chính
        v, nhan = v.clip(-2, 2), f"goc_mat (giá trị ngoài ±2 dồn về biên: {int((v.abs() > 2).sum())} ảnh)"
    else:
        nhan = cot
    ax.hist(v, bins=40, color="#4C72B0")
    ax.set_xlabel(nhan)
    ax.set_ylabel("Số ảnh")
    ax.set_title(ten)
    fig.tight_layout()
    fig.savefig(KQ / file)
    plt.show()

## 4. Trực quan hoá vector đặc trưng

In [ ]:
X_all = df[E].to_numpy(np.float32)
danh_tinh = np.where(df.buoi == "lfw", "lfw_" + df["clip"].astype(str), df.nguoi)
rng = np.random.default_rng(SEED)
idx = rng.choice(len(df), min(1500, len(df)), replace=False)
S = X_all[idx] @ X_all[idx].T
iu = np.triu_indices(len(idx), 1)
cung = danh_tinh[idx][iu[0]] == danh_tinh[idx][iu[1]]
cos = S[iu]
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(cos[~cung], bins=60, alpha=0.6, density=True, label=f"Khác người ({(~cung).sum()} cặp)")
ax.hist(cos[cung], bins=60, alpha=0.6, density=True, label=f"Cùng người ({cung.sum()} cặp)")
ax.axvline(NGUONG_COSINE, color="k", ls="--", label=f"Ngưỡng app = {NGUONG_COSINE}")
ax.set_xlabel("Độ giống cosine")
ax.set_ylabel("Mật độ")
ax.legend()
ax.set_title("Phân bố độ giống cosine giữa các cặp ảnh")
fig.tight_layout()
fig.savefig(KQ / "histogram_cosine.png")
plt.show()
print(f"Cặp khác người: {100 * (cos[~cung] < NGUONG_COSINE).mean():.1f}% dưới {NGUONG_COSINE}; "
      f"cặp cùng người: {100 * (cos[cung] >= NGUONG_COSINE).mean():.1f}% từ {NGUONG_COSINE} trở lên")

In [ ]:
from sklearn.manifold import TSNE

Z = TSNE(n_components=2, metric="cosine", init="pca", random_state=SEED).fit_transform(X_all[idx])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for n in sorted(set(df.nguoi)):
    m = df.nguoi.to_numpy()[idx] == n
    ax.scatter(*Z[m].T, s=6 if n == LA else 10, alpha=0.4 if n == LA else 0.8, label=n, c="lightgray" if n == LA else None)
ax.legend(markerscale=2)
ax.set_title("t-SNE của vector ArcFace 512 chiều")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
fig.savefig(KQ / "tsne.png")
plt.show()